# Healthcare Data Insights and Predictive Modeling
## Predicting Early Hospital Readmission in Diabetic Patients

**Project type:** Internship group project  
**Dataset:** Diabetes 130-US Hospitals for Years 1999-2008  
**Target:** Early readmission within 30 days (`readmitted == '<30'`)

This notebook follows the internship brief: EDA, data cleaning, categorical encoding, class-imbalance handling, classification models, and evaluation.

In [ ]:
!pip -q install ucimlrepo imbalanced-learn

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from ucimlrepo import fetch_ucirepo
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from imblearn.pipeline import Pipeline as ImbPipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from imblearn.over_sampling import SMOTE
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_auc_score, RocCurveDisplay
)


## 1. Load the UCI Dataset

In [ ]:
dataset = fetch_ucirepo(id=296)

X = dataset.data.features.copy()
y_original = dataset.data.targets.copy()

# The UCI package exposes the target separately; for this dataset we use the
# original `readmitted` field when available.
df = pd.concat([X, y_original], axis=1)
df.head()


In [ ]:
print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())
print("\nData types:")
display(df.dtypes)


## 2. Data Cleaning and Target Definition

In [ ]:
# Standardize missing-value markers used by the source dataset
df = df.replace("?", np.nan)

# Create a binary target:
# 1 = readmitted within 30 days
# 0 = not readmitted within 30 days
if "readmitted" not in df.columns:
    raise ValueError("The dataset target column 'readmitted' was not found.")

df["early_readmission"] = (df["readmitted"] == "<30").astype(int)

print(df["early_readmission"].value_counts())
print(df["early_readmission"].value_counts(normalize=True))


In [ ]:
# Remove original target and identifiers from predictors
drop_cols = [c for c in ["readmitted", "encounter_id", "patient_nbr"] if c in df.columns]
model_df = df.drop(columns=drop_cols)

# Replace unknown/invalid gender with missing
if "gender" in model_df.columns:
    model_df["gender"] = model_df["gender"].replace("Unknown/Invalid", np.nan)

X = model_df.drop(columns=["early_readmission"])
y = model_df["early_readmission"]

print("Features:", X.shape[1])
print("Target distribution:")
display(y.value_counts().rename(index={0:"No early readmission",1:"Early readmission"}))


## 3. Exploratory Data Analysis

In [ ]:
print("Missing values (top 20):")
display(X.isna().sum().sort_values(ascending=False).head(20))


In [ ]:
plt.figure(figsize=(7,4))
sns.countplot(x=y)
plt.title("Early Readmission Target Distribution")
plt.xlabel("Early readmission within 30 days (0=No, 1=Yes)")
plt.ylabel("Number of encounters")
plt.show()


In [ ]:
for col in ["age", "gender", "race", "time_in_hospital", "admission_type_id"]:
    if col in X.columns:
        plt.figure(figsize=(8,4))
        if X[col].nunique(dropna=True) <= 15:
            sns.countplot(data=X, x=col, hue=y)
            plt.xticks(rotation=45)
        else:
            sns.histplot(data=X, x=col, hue=y, element="step", stat="density", common_norm=False)
        plt.title(f"{col} by Early Readmission")
        plt.tight_layout()
        plt.show()


## 4. Train/Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print("Training shape:", X_train.shape)
print("Testing shape:", X_test.shape)


## 5. Preprocessing

In [ ]:
numeric_cols = X_train.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_cols = X_train.select_dtypes(exclude=["int64", "float64"]).columns.tolist()

numeric_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipe, numeric_cols),
    ("cat", categorical_pipe, categorical_cols)
])

print("Numeric columns:", len(numeric_cols))
print("Categorical columns:", len(categorical_cols))


## 6. Logistic Regression with SMOTE

In [ ]:
lr_model = ImbPipeline([
    ("preprocessor", preprocessor),
    ("smote", SMOTE(random_state=42)),
    ("classifier", LogisticRegression(max_iter=1000, class_weight=None))
])

lr_model.fit(X_train, y_train)
lr_pred = lr_model.predict(X_test)
lr_prob = lr_model.predict_proba(X_test)[:,1]

lr_results = {
    "Model": "Logistic Regression + SMOTE",
    "Accuracy": accuracy_score(y_test, lr_pred),
    "Precision": precision_score(y_test, lr_pred, zero_division=0),
    "Recall": recall_score(y_test, lr_pred, zero_division=0),
    "F1": f1_score(y_test, lr_pred, zero_division=0),
    "ROC-AUC": roc_auc_score(y_test, lr_prob)
}
lr_results


## 7. Random Forest with SMOTE

In [ ]:
rf_model = ImbPipeline([
    ("preprocessor", preprocessor),
    ("smote", SMOTE(random_state=42)),
    ("classifier", RandomForestClassifier(
        n_estimators=250,
        random_state=42,
        n_jobs=-1,
        class_weight=None
    ))
])

rf_model.fit(X_train, y_train)
rf_pred = rf_model.predict(X_test)
rf_prob = rf_model.predict_proba(X_test)[:,1]

rf_results = {
    "Model": "Random Forest + SMOTE",
    "Accuracy": accuracy_score(y_test, rf_pred),
    "Precision": precision_score(y_test, rf_pred, zero_division=0),
    "Recall": recall_score(y_test, rf_pred, zero_division=0),
    "F1": f1_score(y_test, rf_pred, zero_division=0),
    "ROC-AUC": roc_auc_score(y_test, rf_prob)
}
rf_results


## 8. Model Comparison

In [ ]:
results = pd.DataFrame([lr_results, rf_results])
display(results.sort_values("F1", ascending=False))


## 9. Classification Reports and Confusion Matrices

In [ ]:
print("Logistic Regression")
print(classification_report(y_test, lr_pred, target_names=["No early readmission","Early readmission"]))

print("Random Forest")
print(classification_report(y_test, rf_pred, target_names=["No early readmission","Early readmission"]))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11,4))

sns.heatmap(confusion_matrix(y_test, lr_pred), annot=True, fmt="d", cmap="Blues", ax=axes[0])
axes[0].set_title("Logistic Regression Confusion Matrix")
axes[0].set_xlabel("Predicted"); axes[0].set_ylabel("Actual")

sns.heatmap(confusion_matrix(y_test, rf_pred), annot=True, fmt="d", cmap="Greens", ax=axes[1])
axes[1].set_title("Random Forest Confusion Matrix")
axes[1].set_xlabel("Predicted"); axes[1].set_ylabel("Actual")

plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(7,5))
RocCurveDisplay.from_predictions(y_test, lr_prob, name="Logistic Regression")
RocCurveDisplay.from_predictions(y_test, rf_prob, name="Random Forest")
plt.title("ROC Curves")
plt.show()


## 10. Interpretation

Use the generated metrics to identify the strongest model. Because early readmission is the minority outcome, do not rely on accuracy alone. Discuss precision, recall and F1-score, and explain the practical meaning of false negatives and false positives in a healthcare setting.

**Important:** This model is a predictive analytics exercise and should not be presented as a clinical diagnostic tool.

## 11. Limitations

- The dataset represents historical records from 130 US hospitals during 1999–2008.
- The data contains missing values and sensitive demographic attributes.
- The model may not generalize to current hospitals or different populations.
- Prediction performance does not establish causation.
- Clinical deployment would require external validation, governance and expert oversight.

## 12. Dataset Citation

Clore, J., Cios, K., DeShazo, J., & Strack, B. (2014). *Diabetes 130-US Hospitals for Years 1999-2008*. UCI Machine Learning Repository. DOI: 10.24432/C5230J.